# 05 — Regras de negócio e retry

**Trilha:** [04 ferramentas com funções](04_ferramentas_com_funcoes.ipynb) → **05 regras e retry** → [06 roteamento por enum](06_roteamento_por_enum.ipynb)

**De onde viemos:** desde o [03](../01_basico/03_enums_e_guardrails.ipynb), uma lista com 3 kg de sal para duas pessoas passa no
contrato. O contrato confere **tipos e vocabulário**; regras como "quanto é razoável por pessoa" são **de negócio**. Aqui elas
viram funções simples — e, quando a resposta quebra uma regra, o erro volta para o modelo tentar de novo (**retry com feedback**).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # prettifier.py e logger_general.py ficam em AI_use_cases/
from logger_general import GeneralLogger
from prettifier import show, show_json

In [2]:
import json
from dataclasses import dataclass, field

from pydantic import ValidationError

from contrato import Ingrediente, ListaDeCompras, Unidade, prompt_com_schema, saida_llm

## 1. As regras, como funções

Cada regra é uma função que recebe a lista e devolve uma **lista de problemas** (vazia = tudo certo). Mensagens claras importam:
elas vão ser lidas pelo modelo no retry.

In [3]:
LIMITE_POR_PESSOA = {Unidade.G: 400, Unidade.KG: 0.4, Unidade.ML: 300, Unidade.L: 0.3, Unidade.UN: 4}
PORCOES_MIN, PORCOES_MAX = 1, 20


def regra_quantidade_por_pessoa(lista: ListaDeCompras) -> list[str]:
    problemas = []
    for item in lista.ingredientes:
        limite = LIMITE_POR_PESSOA[item.unidade] * lista.porcoes
        if item.quantidade > limite:
            problemas.append(f"{item.nome}: {item.quantidade:g} {item.unidade} passa do limite de {limite:g} {item.unidade} "
                             f"para {lista.porcoes} porções")
    return problemas


def regra_sem_duplicados(lista: ListaDeCompras) -> list[str]:
    nomes = [item.nome.strip().lower() for item in lista.ingredientes]
    repetidos = sorted({nome for nome in nomes if nomes.count(nome) > 1})
    return [f"ingrediente repetido: {nome}" for nome in repetidos]


REGRAS = [regra_quantidade_por_pessoa, regra_sem_duplicados]


def verificar_regras(lista: ListaDeCompras) -> list[str]:
    return [problema for regra in REGRAS for problema in regra(lista)]

A lista absurda do 03 (*exemplo construído*), agora conferida:

In [4]:
absurda = ListaDeCompras(prato="Pizza margherita", porcoes=2, ingredientes=[
    Ingrediente(nome="sal", quantidade=3, unidade="kg", categoria="mercearia"),
    Ingrediente(nome="muçarela de búfala", quantidade=40, unidade="un", categoria="laticinios"),
    Ingrediente(nome="Sal", quantidade=5, unidade="g", categoria="mercearia"),
])
for problema in verificar_regras(absurda):
    show(f"- {problema}")

- sal: 3 kg passa do limite de 0.8 kg para 2 porções
- muçarela de búfala: 40 un passa do limite de 8 un para 2 porções
- ingrediente repetido: sal


## 2. Validar a entrada antes de gastar uma chamada

Algumas regras dependem só do que o **usuário** pediu. Essas se conferem antes de chamar o modelo — não adianta pedir 50
porções e depois recusar a resposta:

In [5]:
def validar_pedido(prato: str, porcoes: int) -> None:
    if not prato.strip():
        raise ValueError("prato vazio")
    if not PORCOES_MIN <= porcoes <= PORCOES_MAX:
        raise ValueError(f"porções devem estar entre {PORCOES_MIN} e {PORCOES_MAX}, recebido {porcoes}")


try:
    validar_pedido("Pizza margherita", 50)
except ValueError as ex:
    show(f"ValueError: {ex}")

ValueError: porções devem estar entre 1 e 20, recebido 50


## 3. Retry com feedback

`pedir_lista` junta tudo: valida o pedido, chama o modelo, valida o contrato **e** as regras. Se algo falhar, devolve ao modelo
a resposta dele e a lista de erros, e tenta de novo — até `max_tentativas`. Cada tentativa vira um evento no log (`WARNING`
quando falha, `INFO` quando passa).

A função tem **duas saídas**:

- **sucesso:** o `return ResultadoPedido(...)` fica **dentro do laço** e encerra a função na primeira tentativa sem erros;
- **fracasso:** se todas as tentativas falharem, o laço termina e a função levanta `RuntimeError` (não há retorno).

O `ResultadoPedido` é uma dataclass com a lista validada, o número da tentativa que deu certo e os erros de **cada** tentativa
(uma lista por tentativa; a última é `[]`, porque foi a que passou).

In [6]:
INSTRUCAO = "Você gera listas de compras com ingredientes in natura e sistema métrico, em quantidades realistas."


@dataclass
class ResultadoPedido:
    lista: ListaDeCompras
    tentativas: int
    erros_por_tentativa: list[list[str]] = field(default_factory=list)


def erros_da_resposta(texto: str) -> tuple[ListaDeCompras | None, list[str]]:
    try:
        lista = saida_llm(ListaDeCompras).validate_python(texto)   # borda: texto -> objeto, inclusive "sem JSON"
    except ValidationError as ex:
        return None, [f"{'.'.join(map(str, e['loc'])) or 'resposta'}: {e['msg']}" for e in ex.errors()]
    return lista, verificar_regras(lista)


log = GeneralLogger(name="retry", stream=sys.stdout)


def pedir_lista(prato: str, porcoes: int, chamar_modelo, max_tentativas: int = 3) -> ResultadoPedido:
    validar_pedido(prato, porcoes)
    mensagens = [
        {"role": "system", "content": prompt_com_schema(INSTRUCAO, ListaDeCompras)},
        {"role": "user", "content": f"{prato} para {porcoes} pessoas"},
    ]
    historico = []
    for tentativa in range(1, max_tentativas + 1):
        texto = chamar_modelo(mensagens)
        lista, erros = erros_da_resposta(texto)
        historico.append(erros)                                  # erros desta tentativa ([] se passou)
        if not erros:
            log.info(f"tentativa {tentativa}: lista válida ({len(lista.ingredientes)} itens)")
            return ResultadoPedido(lista, tentativa, historico)  # saída 1 (sucesso): encerra a função aqui, no meio do laço
        log.warning(f"tentativa {tentativa}: {'; '.join(erros)}")
        mensagens.append({"role": "assistant", "content": texto})
        mensagens.append({"role": "user", "content": "A resposta tem problemas. Corrija e devolva o JSON completo:\n- "
                                                     + "\n- ".join(erros)})
    # saída 2 (fracasso): só chega aqui se nenhuma tentativa passou
    raise RuntimeError(f"{max_tentativas} tentativas sem resposta válida; últimos erros: {historico[-1]}")

In [7]:
# api: configuração do cliente — lê a chave e o endpoint do .env
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
api_key = os.getenv("openai")
if not api_key:
    raise ValueError("variável 'openai' (chave da API) não encontrada no .env")
openai_base_url = os.getenv("openai_base_url")
if not openai_base_url:
    raise ValueError("variável 'openai_base_url' não encontrada no .env")

AI_MODEL = "gpt-4o-mini"
ai_client = OpenAI(api_key=api_key, base_url=openai_base_url)

In [8]:
# api: pedido real com retry
def chamar_api(mensagens):
    return ai_client.chat.completions.create(model=AI_MODEL, messages=mensagens).choices[0].message.content


resultado = pedir_lista("Pizza margherita", 2, chamar_api)
show(f"tentativas: {resultado.tentativas}")
show_json(resultado.lista.model_dump(mode="json"), title="resultado.lista")

2026-11-12 17:48:38 | INFO     | tentativa 1: lista válida (8 itens)
tentativas: 1
resultado.lista:
{
  "prato": "Pizza Margherita",
  "porcoes": 2,
  "ingredientes": [
    {
      "nome": "Farinha de trigo",
      "quantidade": 250.0,
      "unidade": "g",
      "categoria": "mercearia"
    },
    {
      "nome": "Água",
      "quantidade": 150.0,
      "unidade": "ml",
      "categoria": "mercearia"
    },
    {
      "nome": "Fermento biológico seco",
      "quantidade": 5.0,
      "unidade": "g",
      "categoria": "mercearia"
    },
    {
      "nome": "Sal",
      "quantidade": 5.0,
      "unidade": "g",
      "categoria": "mercearia"
    },
    {
      "nome": "Queijo mozzarella",
      "quantidade": 200.0,
      "unidade": "g",
      "categoria": "laticinios"
    },
    {
      "nome": "Tomate pelado",
      "quantidade": 200.0,
      "unidade": "g",
      "categoria": "hortifruti"
    },
    {
      "nome": "Manjericão fresco",
      "quantidade": 10.0,
      "unidade": "g",
 

Sem chave: um roteiro **construído** em que a primeira resposta exagera e a segunda corrige.

In [7]:
def roteiro(*respostas):
    fila = iter(respostas)
    return lambda mensagens: next(fila)


exagerada = json.dumps({"prato": "Pizza margherita", "porcoes": 2, "ingredientes": [
    {"nome": "farinha de trigo", "quantidade": 2, "unidade": "kg", "categoria": "mercearia"},
    {"nome": "muçarela de búfala", "quantidade": 150, "unidade": "g", "categoria": "laticinios"},
]}, ensure_ascii=False)
corrigida = exagerada.replace('"quantidade": 2, "unidade": "kg"', '"quantidade": 250, "unidade": "g"')

resultado = pedir_lista("Pizza margherita", 2, roteiro(exagerada, corrigida))
show(f"type(resultado).__name__: {type(resultado).__name__}")
show(f"resultado.tentativas: {resultado.tentativas}")
show(f"resultado.erros_por_tentativa: {resultado.erros_por_tentativa}")
show(f"resultado.lista.ingredientes[0]: {resultado.lista.ingredientes[0]!r}")

2026-11-13 17:24:41 | WARNING  | tentativa 1: farinha de trigo: 2 kg passa do limite de 0.8 kg para 2 porções
2026-11-13 17:24:41 | INFO     | tentativa 2: lista válida (2 itens)
type(resultado).__name__: ResultadoPedido
resultado.tentativas: 2
resultado.erros_por_tentativa: [['farinha de trigo: 2 kg passa do limite de 0.8 kg para 2 porções'], []]
resultado.lista.ingredientes[0]: Ingrediente(nome='farinha de trigo', quantidade=250.0, unidade=<Unidade.G: 
'g'>, categoria=<Categoria.MERCEARIA: 'mercearia'>)


## ⚠️ O que ainda quebra

**1. Cada tentativa reenvia a conversa inteira.** O retry não é de graça: a mensagem cresce a cada rodada, e você paga por
tudo de novo. Medindo o tamanho do que é enviado (*roteiro construído*, três respostas ruins):

In [8]:
tamanhos = []


def medindo(respostas):
    proxima = roteiro(*respostas)

    def chamar(mensagens):
        tamanhos.append(sum(len(m["content"]) for m in mensagens))
        return proxima(mensagens)
    return chamar


try:
    pedir_lista("Pizza margherita", 2, medindo([exagerada] * 3))
except RuntimeError as ex:
    show(f"RuntimeError: {str(ex)[:110]}...")
show(f"caracteres enviados por tentativa: {tamanhos}")

2026-11-13 17:24:41 | WARNING  | tentativa 1: farinha de trigo: 2 kg passa do limite de 0.8 kg para 2 porções
2026-11-13 17:24:41 | WARNING  | tentativa 2: farinha de trigo: 2 kg passa do limite de 0.8 kg para 2 porções
2026-11-13 17:24:41 | WARNING  | tentativa 3: farinha de trigo: 2 kg passa do limite de 0.8 kg para 2 porções
RuntimeError: 3 tentativas sem resposta válida; últimos erros: ['farinha de trigo: 2 kg passa do limite de 0.8
kg para 2 por...
caracteres enviados por tentativa: [1162, 1533, 1904]


O avançado mede custo de verdade (tokens e dinheiro) e corta o histórico com orçamento.

**2. As regras ficam fora do contrato — e dá para esquecer de chamá-las.** `ListaDeCompras.model_validate_json` sozinho aceita
a lista absurda; só `pedir_lista` confere as regras. Qualquer outro caminho que crie a lista pula a verificação:

In [9]:
pulou_regras = ListaDeCompras.model_validate_json(exagerada)
show(f"validou sem regras: {pulou_regras.ingredientes[0].quantidade:g} {pulou_regras.ingredientes[0].unidade}")
show(f"verificar_regras(...): {verificar_regras(pulou_regras)}")

validou sem regras: 2 kg
verificar_regras(...): ['farinha de trigo: 2 kg passa do limite de 0.8 kg para 2 porções']


**3. As regras não sabem nada do contexto.** O limite por pessoa é fixo. Mas "razoável" depende do pedido: orçamento do
usuário, o que já tem na despensa, preços do dia. Passar isso para dentro da validação é o assunto do
[07 (avançado)](../03_avancado/07_contrato_e_contexto.ipynb), com o **contexto de validação** do Pydantic.

**4. Retry não conserta pedido ruim.** Se o problema é determinístico (o pedido em si é inválido, ou a regra é impossível de
cumprir), repetir só gasta. Por isso `validar_pedido` vem **antes** da primeira chamada.

Até aqui, toda pergunta era "gere uma lista". Mas o usuário também pergunta preço, pede substituição, conversa fiado. Cada
tipo de pedido precisa de um caminho diferente. ➡️ [06 — roteamento por enum](06_roteamento_por_enum.ipynb)